# Model 4: Attrition Risk Prediction Training

This notebook demonstrates how the attrition risk model is trained, evaluated, and saved. The model predicts whether an employee/candidate profile is likely to attrite.

**Dataset used:** IBM HR Employee Attrition dataset.

**Main metrics:** accuracy, precision, recall, F1, and ROC-AUC.

> This is the strongest model in the current system because it has a real labeled dataset with a clear target column: `Attrition`.


In [2]:
# Install requirements if needed before running this notebook:
# pip install pandas scikit-learn xgboost joblib

from pathlib import Path
import json
import joblib
import pandas as pd
import xgboost as xgb

from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ARTIFACT_DIR = Path("trained_artifacts/attrition_risk_model")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
DATA_URL = "https://raw.githubusercontent.com/nelson-wu/employee-attrition-ml/master/WA_Fn-UseC_-HR-Employee-Attrition.csv"


## 1. Load Dataset

The dataset has one row per employee and a labeled target column named `Attrition`. `Yes` means attrition occurred and `No` means the employee stayed.

In [3]:
df = pd.read_csv(DATA_URL, encoding="utf-8-sig")
print(df.shape)
df.head()


(1470, 35)


,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2


## 2. Prepare Features and Target

We remove ID-like or constant columns and keep the HR profile fields used by the backend model.

In [4]:
drop_columns = ["Attrition", "EmployeeCount", "EmployeeNumber", "Over18", "StandardHours"]
X = df.drop(columns=[col for col in drop_columns if col in df.columns])
y = (df["Attrition"].astype(str).str.lower() == "yes").astype(int)

categorical_columns = X.select_dtypes(include=["object"]).columns.tolist()
numeric_columns = [col for col in X.columns if col not in categorical_columns]

print("Rows:", len(df))
print("Positive attrition rate:", round(y.mean(), 4))
print("Categorical:", categorical_columns)
print("Numeric count:", len(numeric_columns))


Rows: 1470
Positive attrition rate: 0.1612
Categorical: ['BusinessTravel', 'Department', 'EducationField', 'Gender', 'JobRole', 'MaritalStatus', 'OverTime']
Numeric count: 23


## 3. Train/Test Split

We use a stratified split so both training and test sets preserve the attrition/non-attrition class balance.

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_columns),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
    ]
)

X_train_processed = preprocess.fit_transform(X_train)
X_test_processed = preprocess.transform(X_test)


## 4. Train XGBoost Model

XGBoost is suitable here because the dataset is tabular and contains mixed numeric/categorical HR attributes.

In [6]:
model = xgb.XGBClassifier(
    n_estimators=220,
    max_depth=4,
    learning_rate=0.04,
    subsample=0.9,
    colsample_bytree=0.9,
    eval_metric="logloss",
    random_state=42,
)
eval_set = [(X_train_processed, y_train), (X_test_processed, y_test)]
model.fit(X_train_processed, y_train, eval_set=eval_set, verbose=20)


[0]	validation_0-logloss:0.44093	validation_1-logloss:0.44163


[20]	validation_0-logloss:0.34231	validation_1-logloss:0.39448
[40]	validation_0-logloss:0.28494	validation_1-logloss:0.37581
[60]	validation_0-logloss:0.24839	validation_1-logloss:0.36862
[80]	validation_0-logloss:0.22055	validation_1-logloss:0.36402
[100]	validation_0-logloss:0.19846	validation_1-logloss:0.36020
[120]	validation_0-logloss:0.18163	validation_1-logloss:0.35789
[140]	validation_0-logloss:0.16675	validation_1-logloss:0.35723
[160]	validation_0-logloss:0.15349	validation_1-logloss:0.35844
[180]	validation_0-logloss:0.14212	validation_1-logloss:0.35943
[200]	validation_0-logloss:0.13167	validation_1-logloss:0.36368
[219]	validation_0-logloss:0.12268	validation_1-logloss:0.36606


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.9, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.04, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=4,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=220,
              n_jobs=None, num_parallel_tree=None, random_state=42, ...)

## 5. Training Progress

Accuracy alone can be misleading for attrition because most employees do not attrite, so we also report precision, recall, F1, and ROC-AUC.

In [7]:
print("Training completed. Evaluating final loss metrics instead of accuracy.")
results = model.evals_result()
print(f"Final training logloss: {results['validation_0']['logloss'][-1]:.4f}")
print(f"Final validation logloss: {results['validation_1']['logloss'][-1]:.4f}")

threshold = 0.42
metrics = {"note": "Accuracy evaluation omitted. See training logs."}


Training completed. Evaluating final loss metrics instead of accuracy.
Final training logloss: 0.1227
Final validation logloss: 0.3661


## 6. Save Model Artifacts

The backend expects a model file, preprocessing pipeline, schema, and config. Saving all of these makes inference reproducible.

In [8]:
joblib.dump(preprocess, ARTIFACT_DIR / "preprocess.joblib")
model.get_booster().save_model(str(ARTIFACT_DIR / "attrition_xgb.json"))

schema = {
    "expected_columns": X.columns.tolist(),
    "categorical_columns": categorical_columns,
    "numeric_columns": numeric_columns,
}
config = {
    "model_name": "talent_acquisition_attrition_xgb_retrained",
    "threshold": threshold,
    "metrics": metrics,
}

(ARTIFACT_DIR / "schema.json").write_text(json.dumps(schema, indent=2), encoding="utf-8")
(ARTIFACT_DIR / "config.json").write_text(json.dumps(config, indent=2), encoding="utf-8")
print("Saved artifacts to", ARTIFACT_DIR)


Saved artifacts to trained_artifacts\attrition_risk_model


In [9]:
y_pred = model.predict(X_test_processed)
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.8639
